In [3]:
%pip install git+https://github.com/Eickland/HumSpectra.git --upgrade

  Cloning https://github.com/Eickland/HumSpectra.git to c:\users\mnbv2\appdata\local\temp\pip-req-build-7u7hb0nf
  Resolved https://github.com/Eickland/HumSpectra.git to commit 21839d4f794f96898e2485c0eb31a0fa627da75f
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for HumSpectra: filename=humspectra-0.1.37a0-py3-none-any.whl size=45979 sha256=10c325bb9c922cbbcab4a43ee7944e385fe69c6289fb6839b7c9e57ec3edb873
  Stored in directory: C:\Users\mnbv2\AppData\Local\Temp\pip-ephem-wheel-cache-39247eye\wheels\00\54\00\a34c7eb5c74e6384d9a7e88fb33d264b784dfbe30079a3a476
Successfully built HumSpectra
  Attempting uninstall: HumSpectra
    Found existing installation: HumSpectra 0.1.36a0
    Uninstalling HumSpectra-0.1.36a0:
      Successfully uninstalled HumSpectra-0.1.36a0
Note: you may need to restart the kernel to use updated packages.


  Running command git clone --filter=blob:none --quiet https://github.com/Eickland/HumSpectra.git 'C:\Users\mnbv2\AppData\Local\Temp\pip-req-build-7u7hb0nf'
  DEPRECATION: Building 'HumSpectra' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'HumSpectra'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path
from pandas import DataFrame
from typing import List
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
from io import StringIO
import sys
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, 
                             mean_squared_error, r2_score, mean_absolute_error)
warnings.filterwarnings('ignore')
matplotlib.use('Agg')
%matplotlib inline

import HumSpectra.fluorescence as Hfl
import HumSpectra.ultraviolet as Huv
import HumSpectra.math_stat as Hst
import HumSpectra.geomodule as Hgeo
import HumSpectra.drawmodule as Hplt

In [3]:
wdir_2025 = Path.cwd().parent / "Исходные данные" / "2025"

fluo_paths = Path.joinpath(wdir_2025, "Флуоресценция")
uv_paths = Path.joinpath(wdir_2025, "UV","CSV спектры")
fluo_spectra_list = [Hfl.read_fluo_3d(str(path),index_col=1) for path in fluo_paths.rglob('*.csv')]
uv_spectra_list:List = [Huv.read_csv_uv(str(path),spectra_type="absorption") for path in uv_paths.rglob('*.csv')]

toc_data = pd.read_excel(Path.joinpath(wdir_2025, 'TOC.xlsx')) # pyright: ignore[reportCallIssue]
map_data = pd.read_excel(Path.joinpath(wdir_2025, 'Карты.xlsx'))

fl_data = pd.DataFrame()

fl_data["Sample"] = [fl_spectra.attrs['name']
    for fl_spectra in fluo_spectra_list]

fl_data["Class"] = [fl_spectra.attrs['class']
    for fl_spectra in fluo_spectra_list]

fl_data["Subclass"] = [fl_spectra.attrs['subclass']
    for fl_spectra in fluo_spectra_list]

fl_data["Asm350"] = [Hfl.asm_350(fl_spectra)
    for fl_spectra in fluo_spectra_list]

fl_data["Asm280"] = [Hfl.asm_280(fl_spectra)
    for fl_spectra in fluo_spectra_list]

fl_data["FIX"] = [Hfl.fluo_index(fl_spectra)
    for fl_spectra in fluo_spectra_list]

fl_data["HIX"] = [Hfl.humin_index(fl_spectra)
    for fl_spectra in fluo_spectra_list]

fl_data = fl_data[fl_data.Class == "ADOM"]

uv_data = pd.DataFrame()

uv_data["Sample"] = [uv_spectra.attrs['name']
    for uv_spectra in uv_spectra_list]

uv_data["Class"] = [uv_spectra.attrs['class']
    for uv_spectra in uv_spectra_list]

uv_data["Subclass"] = [uv_spectra.attrs['subclass']
    for uv_spectra in uv_spectra_list]

uv_data["E2/E3"] = [Huv.e2_e3(uv_spectra)
    for uv_spectra in uv_spectra_list]

uv_data["E4/E6"] = [Huv.e4_e6(uv_spectra)
    for uv_spectra in uv_spectra_list]

uv_data["lambda"] = [Huv.lambda_UV(uv_spectra)
    for uv_spectra in uv_spectra_list]

uv_data["epsilon"] = [Huv.epsilon(uv_spectra)
    for uv_spectra in uv_spectra_list]

test = uv_data.copy()
uv_data = uv_data.merge(toc_data,on="Sample",how="inner")
uv_data = uv_data[uv_data.Class == "ADOM"]

uv_data["Suva"] = uv_data['epsilon']/uv_data['C_org']
uv_data.drop(columns=["epsilon"],inplace=True)

In [17]:
MainUVData = uv_data.merge(map_data[["Sample","pH","eH"]],on="Sample",how="inner").merge(fl_data, on=["Sample","Class","Subclass"],how="inner")
MainUVData.rename(columns={"E2/E3":"E2E3","E4/E6":"E4E6"},inplace=True)
MainUVData.drop_duplicates(inplace=True)

feature_columns = ["E2E3","C_org","Suva"]
target_list = ["pH","eH"]

MainUVData.sort_values(by="Subclass",inplace=True)
MainUVData.drop(columns=["SD Conc","CV Conc"],inplace=True)
MainUVData.dropna(inplace=True)

MainUVData.to_excel(Path.joinpath(wdir_2025,"MainUVData.xlsx"),index=False)

MainUVData.set_index(["Class","Subclass","Sample"],inplace=True)
MainUVData_ejc = Hst.delete_eject_iqr(MainUVData,iqr_param=3,multi_index=True)

MainUVData_ejc.to_excel(Path.joinpath(wdir_2025,"MainUVData_ejc.xlsx"),index=False)

storage_list = np.unique(MainUVData_ejc.reset_index().Subclass.to_numpy())
storage_index = MainUVData_ejc.reset_index().Subclass.to_numpy()

In [16]:
MainUVData_ejc

E2E3      E4E6    lambda     C_org  \
Class Subclass  Sample                                                  
ADOM  Baikal    ADOM-SLB-BO-2  1.817959  1.000000  0.004838  2.270000   
                ADOM-SLB-3-2   4.138715  1.444346  0.010947  2.335829   
                ADOM-SLB-3-1   2.332741  0.600320  0.003782  2.923000   
                ADOM-SLB-3-3   1.818107  0.500083  0.004273  1.969000   
      Storage 1 ADOM-SL1-9     5.444303  1.999750  0.011242  8.755000   
...                                 ...       ...       ...       ...   
      Storage 9 ADOM-SL9-3     2.847746  1.874781  0.013778  5.191000   
                ADOM-SL9-4     2.893496  1.888593  0.016036  4.763000   
                ADOM-SL9-5     2.661183  1.999667  0.021531  4.609000   
                ADOM-SL9-7     2.658081  1.777475  0.029417  5.684000   
                ADOM-SL9-8     2.432258  1.651919  0.034399  6.126000   

                                   Suva    pH     eH    Asm350    Asm280  \
Class Subclass  Sample                                                     
ADOM  Baikal    ADOM-SLB-BO-2  0.009252  8.50   18.0  5.273624  0.719649   
                ADOM-SLB-3-2   0.071924  8.60  108.0  3.916794  0.724991   
                ADOM-SLB-3-1   0.007528  8.80   82.0  4.675019  0.719291   
                ADOM-SLB-3-3   0.011682  9.60  132.0  5.659382  0.887920   
      Storage 1 ADOM-SL1-9     0.043861  8.48   21.0  4.270170  0.711763   
...                                 ...   ...    ...       ...       ...   
      Storage 9 ADOM-SL9-3     0.027933  7.10  140.0  4.853461  0.976352   
                ADOM-SL9-4     0.031913  6.98  143.0  5.237272  1.017765   
                ADOM-SL9-5     0.039706  7.02  110.0  5.116560  1.059845   
                ADOM-SL9-7     0.040993  7.11  125.0  4.533104  0.855550   
                ADOM-SL9-8     0.032486  5.60   52.0  4.950833  0.803243   

                                    FIX        HIX  
Class Subclass  Sample                              
ADOM  Baikal    ADOM-SLB-BO-2  1.503102  10.886811  
                ADOM-SLB-3-2   1.285343  14.828493  
                ADOM-SLB-3-1   1.450061  14.433937  
                ADOM-SLB-3-3   1.552392  11.611969  
      Storage 1 ADOM-SL1-9     1.325357  17.383773  
...                                 ...        ...  
      Storage 9 ADOM-SL9-3     1.380641  11.482666  
                ADOM-SL9-4     1.440009  11.230290  
                ADOM-SL9-5     1.441570   8.090167  
                ADOM-SL9-7     1.360911  12.251424  
                ADOM-SL9-8     1.392840  14.357164  

[71 rows x 11 columns]

In [ ]:
df = MainUVData_ejc

for feature in feature_columns:

    fig, axes = plt.subplots(1,2,figsize=(12,6))

    for i,target in enumerate(target_list):

        sns.scatterplot(data=df.reset_index(), x=target, y=feature, hue="Subclass",ax=axes[i])

    handles, labels = axes[0].get_legend_handles_labels()
    for ax in axes:
        ax.get_legend().remove()
    fig.legend(handles, labels, loc='center left', ncol=1, 
           bbox_to_anchor=(0.976, 0.779))
    
    fig.suptitle(f"Зависимость {feature} от pH и eH")

    plt.tight_layout()

    plt.savefig(Path.joinpath(wdir_2025,"Зависимость УФ параметров и Corg от pH и eH","Стандарнтый вид",rf"Зависимость {feature} от pH и eH.png"), bbox_inches='tight', pad_inches=0.1)

    plt.show()

In [ ]:
def source_clustering(row):

    Subclass = row.name[1]
    

    if Subclass == "Storage 4" or Subclass == "Storage 7" or Subclass == "Storage 1":
        return "Ash"
    
    elif Subclass == "Baikal" or Subclass == "Storage 8":
        return "Outlier"
    
    else:
        return "Lignin"
    
def acid_clustering(row):

    pH = row['pH']

    if pH < 5.7:
        return "Acid"
    
    elif pH > 7.7:
        return "Alkaline"
    
    else:
        return "Neutral"

def oxidation_clustering(row):

    eH = row['eH']

    if eH < 0:
        return "Reduced"
    
    elif eH > 0:
        return "Oxidized"
    
    else:
        return "Neutral"

In [ ]:
MainUVData_ejc["Source"] = MainUVData_ejc.apply(lambda x: source_clustering(x), axis=1)
MainUVData_ejc["Acidity"] = MainUVData_ejc.apply(lambda x: acid_clustering(x), axis=1)
MainUVData_ejc["Oxidation"] = MainUVData_ejc.apply(lambda x: oxidation_clustering(x), axis=1)

In [ ]:
df = MainUVData_ejc

for feature in feature_columns:

    fig, axes = plt.subplots(1,2,figsize=(12,6))

    for i,target in enumerate(target_list):

        sns.scatterplot(data=df.reset_index(), x=target, y=feature, hue="Source",ax=axes[i])

    handles, labels = axes[0].get_legend_handles_labels()
    for ax in axes:
        ax.get_legend().remove()
    fig.legend(handles, labels, loc='center left', ncol=1, 
           bbox_to_anchor=(0.976, 0.779))
    
    fig.suptitle(f"Зависимость {feature} от pH и eH\nAsh = K4,K7; Outlier = Baikal,K8; Lignin = K2,K3,K4,K5,K6,K9,K10")

    plt.tight_layout()

    plt.savefig(Path.joinpath(wdir,"Полученные данные","2025","Зависимость УФ параметров и Corg от pH и eH","Кластеризация по происхождению",rf"Зависимость {feature} от pH и eH.png")
                , bbox_inches='tight', pad_inches=0.1)

    plt.show()    

In [ ]:
df = MainUVData_ejc

for feature in feature_columns:

    fig, axes = plt.subplots(1,2,figsize=(12,6))

    for i,target in enumerate(target_list):

        sns.scatterplot(data=df.reset_index(), x=target, y=feature, hue="Acidity",ax=axes[i], style="Source")

    handles, labels = axes[0].get_legend_handles_labels()
    for ax in axes:
        ax.get_legend().remove()
    fig.legend(handles, labels, loc='center left', ncol=1, 
           bbox_to_anchor=(0.976, 0.779))
    
    fig.suptitle(f"Зависимость {feature} от pH и eH\nAcid <5.7; Neutral = 5.7-7.7; Alkaline >7,7")

    plt.tight_layout()

    plt.savefig(Path.joinpath(wdir,"Полученные данные","2025","Зависимость УФ параметров и Corg от pH и eH","Кластеризация по кислотности",rf"Зависимость {feature} от pH и eH.png")
                , bbox_inches='tight', pad_inches=0.1)

    plt.show()

In [ ]:
df = MainUVData_ejc

for feature in feature_columns:

    fig, axes = plt.subplots(1,2,figsize=(12,6))

    for i,target in enumerate(target_list):

        sns.scatterplot(data=df.reset_index(), x=target, y=feature, hue="Oxidation",ax=axes[i], style="Source")

    handles, labels = axes[0].get_legend_handles_labels()
    for ax in axes:
        ax.get_legend().remove()
    fig.legend(handles, labels, loc='center left', ncol=1, 
           bbox_to_anchor=(0.976, 0.779))
    
    fig.suptitle(f"Зависимость {feature} от pH и eH\nReduced <0; Neutral = 0-75; Oxidized >75")

    plt.tight_layout()

    plt.savefig(Path.joinpath(wdir,"Полученные данные","2025","Зависимость УФ параметров и Corg от pH и eH","Кластеризация по окисленности","Трехмерная",rf"Зависимость {feature} от pH и eH.png")
                , bbox_inches='tight', pad_inches=0.1)

    plt.show()

K-Means

In [ ]:
def cluster_analysis_with_subclasses(df, n_clusters=None, random_state=42, output_html_path=None):
    """
    Кластеризация данных с многоуровневым индексом и анализ подклассов
    с выводом результатов в HTML файл
    
    Parameters:
    df - DataFrame с многоуровневым индексом
    n_clusters - количество кластеров (если None, подбирается автоматически)
    random_state - для воспроизводимости
    output_html_path - путь для сохранения HTML отчета
    """
    
    # Перехватываем вывод в консоль
    old_stdout = sys.stdout
    sys.stdout = captured_output = StringIO()
    
    try:
        # Сохраняем оригинальные индексы
        original_index = df.index
        subclasses = df.index.get_level_values(1).unique()  # Все уникальные подклассы
        
        print("=" * 60)
        print("АНАЛИЗ КЛАСТЕРИЗАЦИИ С ПОДКЛАССАМИ")
        print("=" * 60)
        
        # Шаг 1: Предобработка данных
        print("1. Предобработка данных...")
        
        # Удаляем нечисловые колонки если есть
        numeric_columns = df.select_dtypes(include=[np.number]).columns
        df_numeric = df[numeric_columns].copy()
        
        # Проверяем наличие пропущенных значений
        if df_numeric.isnull().sum().sum() > 0:
            print(f"   Обнаружено пропущенных значений: {df_numeric.isnull().sum().sum()}")
            df_numeric = df_numeric.fillna(df_numeric.mean())
        
        # Масштабирование признаков
        scaler = StandardScaler()
        data_scaled = scaler.fit_transform(df_numeric)
        
        print(f"   Размерность данных: {data_scaled.shape}")
        print(f"   Количество признаков: {data_scaled.shape[1]}")
        
        # Шаг 2: Определение оптимального числа кластеров (если не задано)
        if n_clusters is None:
            print("2. Подбор оптимального числа кластеров...")
            best_k = 2
            best_silhouette = -1
            
            for k in range(2, min(11, len(df) // 2)):
                kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=10)
                cluster_labels = kmeans.fit_predict(data_scaled)
                silhouette_avg = silhouette_score(data_scaled, cluster_labels)
                
                print(f"   k={k}: Silhouette Score = {silhouette_avg:.4f}")
                
                if silhouette_avg > best_silhouette:
                    best_silhouette = silhouette_avg
                    best_k = k
            
            n_clusters = best_k
            print(f"   Оптимальное число кластеров: {n_clusters} (silhouette: {best_silhouette:.4f})")
        else:
            print(f"2. Используется заданное число кластеров: {n_clusters}")
        
        # Шаг 3: Кластеризация K-Means
        print("3. Выполнение кластеризации K-Means...")
        kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
        cluster_labels = kmeans.fit_predict(data_scaled)
        
        # Добавляем метки кластеров к данным
        result_df = df.copy()
        result_df['cluster'] = cluster_labels
        
        # Шаг 4: Анализ распределения подклассов по кластерам
        print("4. Анализ распределения подклассов по кластерам...")
        
        # Создаем таблицу сопряженности: подклассы vs кластеры
        contingency_table = pd.crosstab(
            index=result_df.index.get_level_values(1),  # подклассы
            columns=result_df['cluster'],
            margins=True,
            margins_name='Всего'
        )
        
        # Процентное распределение по строкам (какой % подкласса попал в каждый кластер)
        percentage_table = pd.crosstab(
            index=result_df.index.get_level_values(1),
            columns=result_df['cluster'],
            normalize='index'
        ) * 100
        
        # Шаг 5: Вывод результатов
        print("\n" + "=" * 60)
        print("РЕЗУЛЬТАТЫ КЛАСТЕРИЗАЦИИ")
        print("=" * 60)
        
        print(f"\nОбщая информация:")
        print(f"- Всего наблюдений: {len(result_df)}")
        print(f"- Количество кластеров: {n_clusters}")
        print(f"- Количество подклассов: {len(subclasses)}")
        print(f"- Silhouette Score: {silhouette_score(data_scaled, cluster_labels):.4f}")
        
        print(f"\nРаспределение по кластерам:")
        cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
        for cluster, count in cluster_counts.items():
            print(f"  Кластер {cluster}: {count} наблюдений ({count/len(result_df)*100:.1f}%)")
        
        print(f"\nТАБЛИЦА СОПРЯЖЕННОСТИ: Подклассы × Кластеры")
        print("-" * 50)
        print(contingency_table)
        
        print(f"\nПРОЦЕНТНОЕ РАСПРЕДЕЛЕНИЕ (% по строкам)")
        print("-" * 50)
        print(percentage_table.round(1))
        
        # Анализ "чистоты" кластеров
        print(f"\nАНАЛИЗ 'ЧИСТОТЫ' КЛАСТЕРОВ:")
        print("-" * 50)
        
        cluster_summary = []
        for cluster in range(n_clusters):
            cluster_data = result_df[result_df['cluster'] == cluster]
            cluster_subclasses = cluster_data.index.get_level_values(1)
            
            if len(cluster_data) > 0:
                # Самый частый подкласс в кластере
                subclass_counts = cluster_subclasses.value_counts()
                top_subclass = subclass_counts.index[0]
                top_count = subclass_counts.iloc[0]
                top_percentage = (top_count / len(cluster_data)) * 100
                
                # Количество уникальных подклассов в кластере
                unique_subclasses = cluster_subclasses.nunique()
                
                cluster_summary.append({
                    'Кластер': cluster,
                    'Наблюдений': len(cluster_data),
                    'Уникальных подклассов': unique_subclasses,
                    'Доминирующий подкласс': top_subclass,
                    'Доля доминирующего': f"{top_percentage:.1f}%"
                })
        
        cluster_summary_df = pd.DataFrame(cluster_summary)
        print(cluster_summary_df.to_string(index=False))
        
        # Дополнительный анализ характеристик кластеров
        print(f"\nХАРАКТЕРИСТИКИ КЛАСТЕРОВ (средние значения):")
        print("-" * 60)
        
        numeric_columns = result_df.select_dtypes(include=[np.number]).columns
        numeric_columns = numeric_columns[numeric_columns != 'cluster']  # Исключаем колонку с кластерами
        
        cluster_means = result_df.groupby('cluster')[numeric_columns].mean()
        print(cluster_means.round(3))
        
        # Маппинг подклассов на основные кластеры
        print(f"\nМАППИНГ ПОДКЛАССОВ НА ОСНОВНЫЕ КЛАСТЕРЫ:")
        print("-" * 50)
        
        subclass_cluster_map = {}
        for subclass in result_df.index.get_level_values(1).unique():
            subclass_data = result_df.xs(subclass, level=1)
            if len(subclass_data) > 0:
                # Определяем основной кластер для подкласса
                main_cluster = subclass_data['cluster'].mode()
                if len(main_cluster) > 0:
                    subclass_cluster_map[subclass] = main_cluster[0]
                    print(f"  {subclass} → Кластер {main_cluster[0]}")
        
        # Получаем весь вывод
        console_output = captured_output.getvalue()
        
        # Восстанавливаем stdout
        sys.stdout = old_stdout
        
        # Создаем HTML отчет
        create_html_report(console_output, result_df, contingency_table, 
                          percentage_table, cluster_summary_df, cluster_means, 
                          subclass_cluster_map, n_clusters, output_html_path)
        
        print(f"\n✅ HTML отчет сохранен в файл: {output_html_path}")
        
        return result_df, kmeans, scaler, contingency_table, percentage_table, subclass_cluster_map
        
    except Exception as e:
        # Восстанавливаем stdout в случае ошибки
        sys.stdout = old_stdout
        print(f"Ошибка при выполнении анализа: {e}")
        raise

def create_html_report(console_output, result_df, contingency_table, percentage_table, 
                      cluster_summary_df, cluster_means, subclass_cluster_map, 
                      n_clusters, output_html_path):
    """Создание HTML отчета с результатами анализа"""
    
    # Создаем стилизованный HTML
    html_content = f"""
    <!DOCTYPE html>
    <html lang="ru">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Анализ кластеризации с подклассами</title>
        <style>
            body {{
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                background-color: #ffffff; /* БЕЛЫЙ ФОН */
            }}
            .container {{
                max-width: 1200px;
                margin: 0 auto;
                background: white;
                padding: 30px;
                border-radius: 10px;
                box-shadow: 0 0 10px rgba(0,0,0,0.05); /* Более легкая тень */
            }}
            .header {{
                background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                color: white;
                padding: 30px;
                border-radius: 10px;
                margin-bottom: 30px;
                text-align: center;
            }}
            .section {{
                margin-bottom: 30px;
                padding: 20px;
                background: #ffffff; /* БЕЛЫЙ ФОН СЕКЦИЙ */
                border-radius: 8px;
                border-left: 4px solid #667eea;
                border: 1px solid #e0e0e0; /* Добавляем границу */
            }}
            .section h2 {{
                color: #333;
                border-bottom: 2px solid #667eea;
                padding-bottom: 10px;
                margin-top: 0;
            }}
            .console-output {{
                background: #f8f9fa; /* Светлый фон вместо темного */
                color: #333; /* Темный текст */
                padding: 20px;
                border-radius: 5px;
                font-family: 'Courier New', monospace;
                white-space: pre-wrap;
                overflow-x: auto;
                font-size: 14px;
                border: 1px solid #ddd;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin: 15px 0;
                background: white;
                border: 1px solid #ddd;
            }}
            table, th, td {{
                border: 1px solid #ddd;
            }}
            th {{
                background: #667eea;
                color: white;
                padding: 12px;
                text-align: left;
            }}
            td {{
                padding: 10px;
                background: white;
            }}
            tr:nth-child(even) {{
                background: #f8f9fa;
            }}
            .highlight {{
                background: #f8f9fa;
                padding: 15px;
                border-radius: 5px;
                border-left: 4px solid #667eea;
            }}
            .summary-grid {{
                display: grid;
                grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
                gap: 15px;
                margin: 20px 0;
            }}
            .summary-card {{
                background: white;
                padding: 15px;
                border-radius: 8px;
                text-align: center;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                border: 1px solid #e0e0e0;
            }}
            .summary-card h3 {{
                margin: 0;
                color: #667eea;
                font-size: 24px;
            }}
            .summary-card p {{
                margin: 5px 0 0 0;
                color: #666;
            }}
            .timestamp {{
                text-align: right;
                color: #888;
                font-style: italic;
                margin-top: 30px;
            }}
            /* Улучшения для таблиц */
            .dataframe {{
                width: 100%;
                border-collapse: collapse;
            }}
            .dataframe th {{
                background: #667eea;
                color: white;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="header">
                <h1>📊 Анализ кластеризации с подклассами</h1>
                <p>Автоматический отчет по результатам кластеризации K-Means</p>
            </div>
            
            <div class="summary-grid">
                <div class="summary-card">
                    <h3>{len(result_df)}</h3>
                    <p>Всего наблюдений</p>
                </div>
                <div class="summary-card">
                    <h3>{n_clusters}</h3>
                    <p>Кластеров</p>
                </div>
                <div class="summary-card">
                    <h3>{len(result_df.index.get_level_values(1).unique())}</h3>
                    <p>Уникальных подклассов</p>
                </div>
                <div class="summary-card">
                    <h3>{result_df.select_dtypes(include=[np.number]).columns.nunique()}</h3>
                    <p>Признаков</p>
                </div>
            </div>
            
            <div class="section">
                <h2>📋 Полный вывод анализа</h2>
                <div class="console-output">{console_output}</div>
            </div>
            
            <div class="section">
                <h2>🔍 Таблица сопряженности: Подклассы × Кластеры</h2>
                {contingency_table.to_html(classes='dataframe', border=0, index=True)}
            </div>
            
            <div class="section">
                <h2>📈 Процентное распределение подклассов</h2>
                <p><em>Проценты показывают распределение каждого подкласса по кластерам</em></p>
                {percentage_table.round(1).to_html(classes='dataframe', border=0, index=True)}
            </div>
            
            <div class="section">
                <h2>🎯 Анализ чистоты кластеров</h2>
                {cluster_summary_df.to_html(classes='dataframe', border=0, index=False)}
            </div>
            
            <div class="section">
                <h2>📊 Характеристики кластеров</h2>
                <p><em>Средние значения признаков по кластерам</em></p>
                {cluster_means.round(3).to_html(classes='dataframe', border=0, index=True)}
            </div>
            
            <div class="section">
                <h2>🗺️ Маппинг подклассов на кластеры</h2>
                <div class="highlight">
    """
    
    # Добавляем маппинг подклассов
    for subclass, cluster in subclass_cluster_map.items():
        html_content += f"<p><strong>{subclass}</strong> → Основной кластер <strong>{cluster}</strong></p>\n"
    
    html_content += f"""
                </div>
            </div>
            
            <div class="timestamp">
                Отчет сгенерирован: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
            </div>
        </div>
    </body>
    </html>
    """
    
    # Сохраняем HTML файл
    with open(output_html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

def analyze_cluster_characteristics(result_df, n_clusters):
    """Анализ характеристик кластеров"""
    print(f"\nХАРАКТЕРИСТИКИ КЛАСТЕРОВ (средние значения):")
    print("-" * 60)
    
    numeric_columns = result_df.select_dtypes(include=[np.number]).columns
    numeric_columns = numeric_columns[numeric_columns != 'cluster']  # Исключаем колонку с кластерами
    
    cluster_means = result_df.groupby('cluster')[numeric_columns].mean()
    print(cluster_means.round(3))

def get_subclass_cluster_mapping(result_df):
    """Получить маппинг подклассов на основные кластеры"""
    subclass_cluster_map = {}
    
    for subclass in result_df.index.get_level_values(1).unique():
        subclass_data = result_df.xs(subclass, level=1)
        if len(subclass_data) > 0:
            # Определяем основной кластер для подкласса
            main_cluster = subclass_data['cluster'].mode()
            if len(main_cluster) > 0:
                subclass_cluster_map[subclass] = main_cluster[0]
    
    return subclass_cluster_map

Выявление классов-outlier

In [ ]:
result_df, kmeans, scaler, contingency_table, percentage_table, subclass_cluster_map = cluster_analysis_with_subclasses(MainUVData_ejc[feature_columns],output_html_path=Path.joinpath(wdir,"Полученные данные","2025","Кластеризация без PCA","Только важные","Выявление классов-аутсайдеров.html"))

Кластеризация без outlier с KMeans

In [ ]:
MainUVData_ejc_filtered = MainUVData_ejc.drop('Baikal', level='Subclass')
MainUVData_ejc_filtered = MainUVData_ejc_filtered.drop('Storage 8', level='Subclass')
MainUVData_ejc_filtered = MainUVData_ejc_filtered.drop('Storage 1', level='Subclass')
result_df, kmeans, scaler, contingency_table, percentage_table, subclass_cluster_map = cluster_analysis_with_subclasses(MainUVData_ejc_filtered[feature_columns],n_clusters=2
                                                                                                                        ,output_html_path=Path.joinpath(wdir,"Полученные данные","2025","Кластеризация без PCA","Только важные","KMeans без аутсайдеров с двумя кластерами.html"))


Кластеризация с Kmeans с pH

In [ ]:
feature_columns_pH = feature_columns + ['pH']

result_df, kmeans, scaler, contingency_table, percentage_table, subclass_cluster_map = cluster_analysis_with_subclasses(
    MainUVData_ejc_filtered[feature_columns_pH],n_clusters=2,
    output_html_path=Path.joinpath(wdir,"Полученные данные","2025","Кластеризация без PCA","Только важные","KMeans без аутсайдеров с двумя кластерами с pH.html")
    )

Кластеризация с Kmeans с pH и eH

In [ ]:
feature_columns_pH_eH = feature_columns + ['pH'] + ['eH']

result_df, kmeans, scaler, contingency_table, percentage_table, subclass_cluster_map = cluster_analysis_with_subclasses(
    MainUVData_ejc_filtered[feature_columns_pH_eH],n_clusters=2,
    output_html_path=Path.joinpath(wdir,"Полученные данные","2025","Кластеризация без PCA","Только важные","KMeans без аутсайдеров с двумя кластерами с pH и с eH.html")
    )

PCA кластеризация

In [ ]:
# 1. Загрузка и предобработка

data = MainUVData_ejc[feature_columns]
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)

# 2. (Опционально) Снижение размерности
pca = PCA(n_components=0.95) # Оставляем компоненты, объясняющие 95% дисперсии
data_pca = pca.fit_transform(data_scaled)

# 3. Подбор числа кластеров для K-Means
inertia = []
sil_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    cluster_labels = kmeans.fit_predict(data_scaled) # или data_pca
    inertia.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(data_scaled, cluster_labels))

# Строим графики Elbow Method и Silhouette Score
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(K_range, inertia, 'bx-')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow Method')

plt.subplot(1,2,2)
plt.plot(K_range, sil_scores, 'rx-')
plt.xlabel('k')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis')
plt.show()

# 4. Запуск финального алгоритма
optimal_k = 4 # Выбрали на основе графиков
final_kmeans = KMeans(n_clusters=optimal_k, random_state=42)
final_clusters = final_kmeans.fit_predict(data_scaled)

# 5. Интерпретация
data['Cluster'] = final_clusters
cluster_profile = data.groupby('Cluster').mean()
#print(cluster_profile)

# 6. Визуализация через PCA (обязательно для 10+ признаков)

fig,axes = plt.subplots(1,2,figsize=(14,6))

pca_for_plot = PCA(n_components=2)
data_2d = pca_for_plot.fit_transform(data_scaled)
axes[0].scatter(data_2d[:, 0], data_2d[:, 1], c=final_clusters, cmap='rainbow')
axes[0].set_xlabel('First Principal Component')
axes[0].set_ylabel('Second Principal Component')
axes[0].set_title('Clusters Visualized in 2D PCA')

sns.scatterplot(x=data_2d[:, 0], y=data_2d[:, 1], hue=data.reset_index().Subclass, palette='rainbow',ax=axes[1])
axes[1].set_title('PCA без кластеризации')
axes[1].set_xlabel('First Principal Component')

plt.tight_layout()
plt.show()

In [ ]:
def random_forest_analysis_with_subclasses(df, target_column=None, problem_type='auto', 
                                         test_size=0.2, random_state=42, 
                                         output_html_path='random_forest_analysis_report.html',
                                         n_estimators=100, max_depth=None):
    """
    Анализ данных с помощью Random Forest и анализ подклассов
    с выводом результатов в HTML файл
    
    Parameters:
    df - DataFrame с многоуровневым индексом
    target_column - название целевой переменной (если None, используется второй уровень индекса)
    problem_type - 'classification', 'regression' или 'auto'
    test_size - доля тестовой выборки
    random_state - для воспроизводимости
    output_html_path - путь для сохранения HTML отчета
    n_estimators - количество деревьев в лесу
    max_depth - максимальная глубина деревьев
    """
    
    # Перехватываем вывод в консоль
    old_stdout = sys.stdout
    sys.stdout = captured_output = StringIO()
    
    try:
        print("=" * 70)
        print("АНАЛИЗ С ПОМОЩЬЮ RANDOM FOREST")
        print("=" * 70)
        
        # Шаг 1: Подготовка данных
        print("1. Подготовка данных...")
        
        # Определяем целевую переменную
        if target_column is None:
            # Используем второй уровень индекса как целевую переменную
            target = df.index.get_level_values(1)
            features_df = df.reset_index(drop=True)
            target_name = "Первый уровень индекса"
        else:
            target = df[target_column]
            features_df = df.drop(columns=[target_column])
            target_name = target_column
        
        # Определяем тип задачи
        if problem_type == 'auto':
            if target.dtype == 'object' or target.nunique() < 20:
                problem_type = 'classification'
            else:
                problem_type = 'regression'
        
        print(f"   Тип задачи: {problem_type.upper()}")
        print(f"   Целевая переменная: {target_name}")
        print(f"   Уникальных значений в целевой переменной: {target.nunique()}")
        
        # Кодируем целевую переменную для классификации
        if problem_type == 'classification':
            le = LabelEncoder()
            target_encoded = le.fit_transform(target)
            class_names = le.classes_
            print(f"   Классы: {list(class_names)}")
        else:
            target_encoded = target
            le = None
            class_names = None
        
        # Шаг 2: Предобработка признаков
        print("2. Предобработка признаков...")
        
        # Удаляем нечисловые колонки если есть
        numeric_columns = features_df.select_dtypes(include=[np.number]).columns
        features_numeric = features_df[numeric_columns].copy()
        
        # Проверяем наличие пропущенных значений
        if features_numeric.isnull().sum().sum() > 0:
            missing_count = features_numeric.isnull().sum().sum()
            print(f"   Обнаружено пропущенных значений: {missing_count}")
            features_numeric = features_numeric.fillna(features_numeric.mean())
        
        # Масштабирование признаков (опционально для Random Forest)
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features_numeric)
        
        print(f"   Размерность данных: {features_scaled.shape}")
        print(f"   Количество признаков: {features_scaled.shape[1]}")
        
        # Шаг 3: Разделение на train/test
        print("3. Разделение на обучающую и тестовую выборки...")
        
        X_train, X_test, y_train, y_test, indices_train, indices_test = train_test_split(
            features_scaled, target_encoded, features_df.index, 
            test_size=test_size, random_state=random_state, stratify=target_encoded if problem_type == 'classification' else None
        )
        
        print(f"   Обучающая выборка: {X_train.shape[0]} наблюдений")
        print(f"   Тестовая выборка: {X_test.shape[0]} наблюдений")
        
        # Шаг 4: Обучение Random Forest
        print("4. Обучение Random Forest...")
        
        if problem_type == 'classification':
            rf_model = RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                random_state=random_state,
                n_jobs=-1
            )
        else:
            rf_model = RandomForestRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                random_state=random_state,
                n_jobs=-1
            )
        
        rf_model.fit(X_train, y_train)
        
        # Шаг 5: Предсказания и оценка модели
        print("5. Оценка модели...")
        
        y_pred = rf_model.predict(X_test)
        y_pred_proba = rf_model.predict(X_test) if problem_type == 'classification' else None
        
        # Метрики в зависимости от типа задачи
        if problem_type == 'classification':
            accuracy = accuracy_score(y_test, y_pred)
            print(f"   Accuracy: {accuracy:.4f}")
            print(f"\n   Classification Report:")
            print("   " + "-" * 50)
            clf_report = classification_report(y_test, y_pred, target_names=class_names, output_dict=True)
            clf_report_df = pd.DataFrame(clf_report).transpose()
            print(clf_report_df.to_string(float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
        else:
            mse = mean_squared_error(y_test, y_pred)
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)
            print(f"   Mean Squared Error: {mse:.4f}")
            print(f"   Mean Absolute Error: {mae:.4f}")
            print(f"   R² Score: {r2:.4f}")
        
        # Шаг 6: Анализ важности признаков
        print("\n6. Анализ важности признаков...")
        
        feature_importance = pd.DataFrame({
            'feature': numeric_columns,
            'importance': rf_model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print("   Топ-10 самых важных признаков:")
        for i, row in feature_importance.head(10).iterrows():
            print(f"      {row['feature']}: {row['importance']:.4f}")
        
        # Шаг 7: Анализ подклассов
        print("\n7. Анализ распределения подклассов...")
        
        # Создаем DataFrame с результатами
        results_df = features_df.copy()
        results_df['actual'] = target_encoded
        results_df['predicted'] = rf_model.predict(features_scaled)
        results_df['is_correct'] = (results_df['actual'] == results_df['predicted']) if problem_type == 'classification' else None
        
        # Анализ точности по подклассам
        if problem_type == 'classification' and df.index.nlevels > 1:
            subclass_accuracy = {}
            subclasses = df.index.get_level_values(1).unique()
            
            for subclass in subclasses:
                subclass_mask = df.index.get_level_values(1) == subclass
                if subclass_mask.sum() > 0:
                    subclass_actual = results_df.loc[subclass_mask, 'actual']
                    subclass_pred = results_df.loc[subclass_mask, 'predicted']
                    acc = accuracy_score(subclass_actual, subclass_pred)
                    subclass_accuracy[subclass] = acc
            
            subclass_acc_df = pd.DataFrame.from_dict(subclass_accuracy, orient='index', columns=['Accuracy'])
            subclass_acc_df = subclass_acc_df.sort_values('Accuracy', ascending=False)
            
            print(f"\n   Точность по подклассам:")
            print("   " + "-" * 40)
            for subclass, acc in subclass_acc_df.head(10).iterrows():
                print(f"      {subclass}: {acc['Accuracy']:.4f}")
        
        # Шаг 8: Анализ ошибок
        print("\n8. Анализ ошибок...")
        
        if problem_type == 'classification':
            cm = confusion_matrix(y_test, y_pred)
            print(f"   Матрица ошибок (Confusion Matrix):")
            print("   " + str(cm))
            
            # Анализ самых частых ошибок
            error_analysis = pd.DataFrame({
                'actual': y_test,
                'predicted': y_pred
            })
            if le is not None:
                error_analysis['actual'] = le.inverse_transform(error_analysis['actual'])
                error_analysis['predicted'] = le.inverse_transform(error_analysis['predicted'])
            
            common_errors = error_analysis[error_analysis['actual'] != error_analysis['predicted']]
            if len(common_errors) > 0:
                error_patterns = common_errors.groupby(['actual', 'predicted']).size().reset_index(name='count')
                error_patterns = error_patterns.sort_values('count', ascending=False)
                print(f"\n   Самые частые ошибки классификации:")
                for _, row in error_patterns.head(5).iterrows():
                    print(f"      {row['actual']} → {row['predicted']}: {row['count']} случаев")
        
        # Получаем весь вывод
        console_output = captured_output.getvalue()
        
        # Восстанавливаем stdout
        sys.stdout = old_stdout
        
        # Создаем HTML отчет
        create_rf_html_report(console_output, results_df, feature_importance, 
                             rf_model, problem_type, class_names, le,
                             X_test, y_test, y_pred, output_html_path)
        
        print(f"\n✅ HTML отчет сохранен в файл: {output_html_path}")
        
        return (results_df, rf_model, scaler, feature_importance, 
                X_train, X_test, y_train, y_test, le)
        
    except Exception as e:
        # Восстанавливаем stdout в случае ошибки
        sys.stdout = old_stdout
        print(f"Ошибка при выполнении анализа: {e}")
        raise

def create_rf_html_report(console_output, results_df, feature_importance, 
                         rf_model, problem_type, class_names, label_encoder,
                         X_test, y_test, y_pred, output_html_path):
    """Создание HTML отчета с результатами анализа Random Forest"""
    
    # Создаем визуализации
    plt.style.use('default')
    
    # График важности признаков
    fig1, ax1 = plt.subplots(figsize=(10, 6))
    top_features = feature_importance.head(15)
    ax1.barh(top_features['feature'], top_features['importance'])
    ax1.set_xlabel('Важность признака')
    ax1.set_title('Топ-15 самых важных признаков')
    ax1.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    
    # Сохраняем график в base64
    from io import BytesIO
    import base64
    
    buffer1 = BytesIO()
    fig1.savefig(buffer1, format='png', dpi=100, bbox_inches='tight')
    buffer1.seek(0)
    feature_importance_plot = base64.b64encode(buffer1.getvalue()).decode()
    plt.close(fig1)
    
    # Матрица ошибок для классификации
    confusion_matrix_plot = ""
    if problem_type == 'classification':
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y_test, y_pred)
        fig2, ax2 = plt.subplots(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                   xticklabels=class_names, yticklabels=class_names)
        ax2.set_xlabel('Предсказанные')
        ax2.set_ylabel('Фактические')
        ax2.set_title('Матрица ошибок')
        plt.tight_layout()
        
        buffer2 = BytesIO()
        fig2.savefig(buffer2, format='png', dpi=100, bbox_inches='tight')
        buffer2.seek(0)
        confusion_matrix_plot = base64.b64encode(buffer2.getvalue()).decode()
        plt.close(fig2)
    
    # Создаем стилизованный HTML
    html_content = f"""
    <!DOCTYPE html>
    <html lang="ru">
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <title>Анализ Random Forest</title>
        <style>
            body {{
                font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
                line-height: 1.6;
                margin: 0;
                padding: 20px;
                background-color: #ffffff;
            }}
            .container {{
                max-width: 1400px;
                margin: 0 auto;
                background: white;
                padding: 30px;
                border-radius: 10px;
                box-shadow: 0 0 10px rgba(0,0,0,0.05);
            }}
            .header {{
                background: linear-gradient(135deg, #28a745 0%, #20c997 100%);
                color: white;
                padding: 30px;
                border-radius: 10px;
                margin-bottom: 30px;
                text-align: center;
            }}
            .section {{
                margin-bottom: 30px;
                padding: 20px;
                background: #ffffff;
                border-radius: 8px;
                border-left: 4px solid #28a745;
                border: 1px solid #e0e0e0;
            }}
            .section h2 {{
                color: #333;
                border-bottom: 2px solid #28a745;
                padding-bottom: 10px;
                margin-top: 0;
            }}
            .console-output {{
                background: #f8f9fa;
                color: #333;
                padding: 20px;
                border-radius: 5px;
                font-family: 'Courier New', monospace;
                white-space: pre-wrap;
                overflow-x: auto;
                font-size: 14px;
                border: 1px solid #ddd;
            }}
            table {{
                width: 100%;
                border-collapse: collapse;
                margin: 15px 0;
                background: white;
                border: 1px solid #ddd;
            }}
            table, th, td {{
                border: 1px solid #ddd;
            }}
            th {{
                background: #28a745;
                color: white;
                padding: 12px;
                text-align: left;
            }}
            td {{
                padding: 10px;
                background: white;
            }}
            tr:nth-child(even) {{
                background: #f8f9fa;
            }}
            .highlight {{
                background: #f8f9fa;
                padding: 15px;
                border-radius: 5px;
                border-left: 4px solid #28a745;
            }}
            .summary-grid {{
                display: grid;
                grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
                gap: 15px;
                margin: 20px 0;
            }}
            .summary-card {{
                background: white;
                padding: 15px;
                border-radius: 8px;
                text-align: center;
                box-shadow: 0 2px 5px rgba(0,0,0,0.1);
                border: 1px solid #e0e0e0;
            }}
            .summary-card h3 {{
                margin: 0;
                color: #28a745;
                font-size: 24px;
            }}
            .summary-card p {{
                margin: 5px 0 0 0;
                color: #666;
            }}
            .plot-container {{
                text-align: center;
                margin: 20px 0;
            }}
            .plot-container img {{
                max-width: 100%;
                height: auto;
                border: 1px solid #ddd;
                border-radius: 5px;
            }}
            .timestamp {{
                text-align: right;
                color: #888;
                font-style: italic;
                margin-top: 30px;
            }}
            .model-info {{
                background: #e8f5e8;
                padding: 15px;
                border-radius: 5px;
                margin: 15px 0;
            }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="header">
                <h1>🌲 Анализ Random Forest</h1>
                <p>Автоматический отчет по результатам анализа с помощью Random Forest</p>
            </div>
            
            <div class="summary-grid">
                <div class="summary-card">
                    <h3>{len(results_df)}</h3>
                    <p>Всего наблюдений</p>
                </div>
                <div class="summary-card">
                    <h3>{len(feature_importance)}</h3>
                    <p>Признаков</p>
                </div>
                <div class="summary-card">
                    <h3>{rf_model.n_estimators}</h3>
                    <p>Деревьев в лесу</p>
                </div>
                <div class="summary-card">
                    <h3>{problem_type.upper()}</h3>
                    <p>Тип задачи</p>
                </div>
            </div>
            
            <div class="model-info">
                <h3>📊 Информация о модели</h3>
                <p><strong>Тип модели:</strong> Random Forest {'Classifier' if problem_type == 'classification' else 'Regressor'}</p>
                <p><strong>Количество деревьев:</strong> {rf_model.n_estimators}</p>
                <p><strong>Максимальная глубина:</strong> {rf_model.max_depth if rf_model.max_depth else 'Не ограничена'}</p>
                <p><strong>Количество признаков:</strong> {rf_model.n_features_in_}</p>
            </div>
            
            <div class="section">
                <h2>📋 Полный вывод анализа</h2>
                <div class="console-output">{console_output}</div>
            </div>
            
            <div class="section">
                <h2>🔍 Важность признаков</h2>
                <p><em>Топ-15 самых важных признаков по версии Random Forest</em></p>
                <div class="plot-container">
                    <img src="data:image/png;base64,{feature_importance_plot}" alt="Feature Importance">
                </div>
                {feature_importance.head(20).to_html(classes='dataframe', border=0, index=False)}
            </div>
    """
    
    # Добавляем матрицу ошибок для классификации
    if problem_type == 'classification' and confusion_matrix_plot:
        html_content += f"""
            <div class="section">
                <h2>🎯 Матрица ошибок</h2>
                <p><em>Визуализация правильных и неправильных предсказаний</em></p>
                <div class="plot-container">
                    <img src="data:image/png;base64,{confusion_matrix_plot}" alt="Confusion Matrix">
                </div>
            </div>
        """
    
    html_content += f"""
            <div class="section">
                <h2>📈 Детальная информация о признаках</h2>
                <p><em>Полная таблица важности всех признаков</em></p>
                {feature_importance.to_html(classes='dataframe', border=0, index=False)}
            </div>
            
            <div class="timestamp">
                Отчет сгенерирован: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
            </div>
        </div>
    </body>
    </html>
    """
    
    # Сохраняем HTML файл
    with open(output_html_path, 'w', encoding='utf-8') as f:
        f.write(html_content)

# ДОПОЛНИТЕЛЬНЫЕ ФУНКЦИИ ДЛЯ УГЛУБЛЕННОГО АНАЛИЗА

def analyze_feature_relationships(rf_model, feature_names, top_n=10):
    """Анализ взаимосвязей признаков"""
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("Топ-{} самых важных признаков:".format(top_n))
    for i, row in importance_df.head(top_n).iterrows():
        print(f"  {row['feature']}: {row['importance']:.4f}")
    
    return importance_df

In [ ]:
results_df, rf_model, scaler, feature_importance, X_train, X_test, y_train, y_test, le = random_forest_analysis_with_subclasses(
    MainUVData_ejc[["C_org","Suva","E2E3","pH"]],
    problem_type='classification',
    output_html_path=str(Path.joinpath(wdir,"Полученные данные","2025","Кластеризация без PCA","Все признаки","Random Forest с аутсайдерами с pH и с eH.html"))
)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred = rf_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='g', cmap='Blues')
plt.title("Матрица ошибок Random Forest")
plt.xlabel('Предсказанные метки')
plt.ylabel('Истинные метки')
plt.show()